In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Numbeo Cost of Living Scraper - Capoluoghi di provincia italiani
===================================================================

COSA FA
-------
Scarica, una pagina alla volta, i dati di "Cost of Living" da numbeo.com
per i capoluoghi delle 110 province italiane (configurazione storica a
110 province) e produce due file CSV:

  - numbeo_italia_long.csv  -> formato "lungo": una riga per ogni singolo
                                prezzo (provincia, categoria, voce, prezzo, range)
  - numbeo_italia_wide.csv  -> formato "tabellare": una riga per provincia,
                                una colonna per ogni voce di prezzo
                                (es. "Restaurants | Cappuccino (Regular Size)")

IMPORTANTE - LEGGERE PRIMA DI USARE
------------------------------------
I Termini di Uso di Numbeo (numbeo.com/common/terms_of_use.jsp) vietano
lo scraping/crawling automatico senza permesso scritto, ma consentono
l'uso libero dei DATI per scopi personali, a patto di citare la fonte
(link a numbeo.com). Questo script è pensato quindi per:
  - uso PERSONALE (non commerciale, non ridistribuzione dei dati grezzi);
  - un ritmo di richieste deliberatamente lento (una richiesta alla volta,
    pausa di alcuni secondi tra una città e l'altra, niente parallelismo);
  - citare sempre numbeo.com come fonte se pubblichi qualcosa basato su
    questi dati.
Per usi commerciali, massivi o ripetuti nel tempo, Numbeo vende un Data
License / API ufficiale: numbeo.com/common/api.jsp

REQUISITI
---------
    pip install requests beautifulsoup4 pandas

USO
---
    python numbeo_scraper_province_italiane.py
"""

import re
import time
import random
import unicodedata
import difflib
from pathlib import Path
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.numbeo.com"
COUNTRY_URL = f"{BASE}/cost-of-living/country_result.jsp?country=Italy"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0 Safari/537.36 personal-research-script"
}

# Pausa (in secondi) tra una richiesta e l'altra: mantienila alta per
# rispettare il server. Non ridurla per "velocizzare" lo scraping.
DELAY_RANGE = (4, 8)

# Sotto questa soglia di voci trovate, la provincia viene segnalata a fine
# esecuzione come "dati incompleti" — non è un errore dello script, significa
# solo che su Numbeo quella città ha poche informazioni inserite dagli utenti.
PARTIAL_THRESHOLD = 30

# ---------------------------------------------------------------------------
# 110 capoluoghi di provincia italiani (configurazione storica a 110 province,
# comprese le 4 province sarde poi riassorbite). Modifica liberamente questa
# lista se vuoi usare la configurazione attuale (107 enti) o un sottoinsieme.
# ---------------------------------------------------------------------------
PROVINCE_CAPITALS = [
    "Verbania",        # Verbano-Cusio-Ossola
        "Imperia",
        "La Spezia",
        "Sondrio",
        "Rovigo",
        "Udine",
        "Massa",           # Massa-Carrara
        "Pesaro",          # Pesaro e Urbino
        "Ascoli Piceno",
        "Viterbo",
        "Frosinone",
        "Campobasso",
        "Isernia",
        "Benevento",
        "Avellino",
        "Potenza",
        "Matera",
        "Catanzaro",
        "Crotone",
        "Vibo Valentia",   # attenzione: probabilmente senza pagina su Numbeo
        "Trapani",
        "Messina",
        "Agrigento",
        "Caltanissetta",   # attenzione: probabilmente senza pagina su Numbeo
        "Enna",            # attenzione: probabilmente senza pagina su Numbeo
        "Ragusa",
        "Siracusa",
        "Nuoro",
        "Oristano",
]

# Alias manuali per i casi in cui Numbeo usa un nome diverso da quello
# "ufficiale" italiano (nomi inglesi per le città maggiori, ecc.)
MANUAL_ALIASES = {
    "torino": "Turin",
    "milano": "Milan",
    "firenze": "Florence",
    "genova": "Genoa",
    "napoli": "Naples",
    "roma": "Rome",
    "venezia": "Venice",
    "reggio emilia": "Reggio Nell'emilia",
    "reggio calabria": "Reggio Di Calabria",
    "bolzano": "Bolzano-Bozen",
}

# Le 10 categorie di prezzo che ci interessano (nomi esatti come compaiono
# sulla pagina di Numbeo)
CATEGORIES = [
    "Restaurants", "Markets", "Transportation", "Utilities (Monthly)",
    "Sports And Leisure", "Childcare", "Clothing And Shoes",
    "Rent Per Month", "Buy Apartment Price", "Salaries And Financing",
]

PRICE_RE = re.compile(r"([€$£])\s*([\d.,]+)")
NUMBER_RE = re.compile(r"[\d][\d.,]*")


def normalize(s: str) -> str:
    """Normalizza un nome città per il confronto: minuscolo, senza accenti,
    solo lettere/numeri."""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", s.lower())


def get_city_slug_map(session: requests.Session) -> dict:
    """Scarica la pagina 'Italy' di Numbeo e legge il menu a tendina delle
    città per ottenere la mappa {nome_normalizzato: slug_url_reale}.
    Questo evita di dover indovinare a mano l'URL di ogni città."""
    r = session.get(COUNTRY_URL, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    select = None
    for sel in soup.find_all("select"):
        opts = sel.find_all("option")
        if opts and "select city" in opts[0].get_text(strip=True).lower():
            select = sel
            break
    if select is None:
        raise RuntimeError(
            "Non trovo il menu a tendina delle città nella pagina Italia. "
            "La struttura del sito potrebbe essere cambiata."
        )

    mapping = {}
    for opt in select.find_all("option"):
        label = opt.get_text(strip=True)
        if not label or "select city" in label.lower():
            continue
        value = (opt.get("value") or label).strip()
        mapping[normalize(label)] = value
    return mapping


def resolve_slug(city_name: str, slug_map: dict):
    """Trova lo slug URL corretto per una data città."""
    alias = MANUAL_ALIASES.get(city_name.lower())
    key = normalize(alias) if alias else normalize(city_name)
    if key in slug_map:
        return slug_map[key]
    close = difflib.get_close_matches(key, slug_map.keys(), n=1, cutoff=0.8)
    if close:
        return slug_map[close[0]]
    return None


def fetch_city_page(session: requests.Session, slug: str):
    url = f"{BASE}/cost-of-living/in/{quote(slug, safe='')}"
    try:
        r = session.get(url, headers=HEADERS, timeout=30)
    except requests.RequestException as e:
        print(f"errore di rete ({e})")
        return None
    if r.status_code != 200:
        print(f"HTTP {r.status_code}")
        return None
    return r.text


def parse_price(text: str):
    """Estrae (valuta, valore) da un testo tipo '€18.00'. Alcune voci
    (es. il tasso di interesse ipotecario) non hanno simbolo di valuta,
    in quel caso ritorna (None, valore)."""
    m = PRICE_RE.search(text)
    if m:
        try:
            return m.group(1), float(m.group(2).replace(",", ""))
        except ValueError:
            return m.group(1), None
    m2 = NUMBER_RE.search(text)
    if m2:
        try:
            return None, float(m2.group(0).replace(",", ""))
        except ValueError:
            return None, None
    return None, None


def find_price_table(soup: BeautifulSoup):
    """Individua la tabella dei prezzi. Numbeo la marca con la classe
    'data_wide_table'; in fallback cerca la tabella che contiene il
    maggior numero di intestazioni di categoria note (utile se cambiano
    le classi CSS in futuro)."""
    table = soup.find("table", class_="data_wide_table")
    if table is not None:
        return table

    best_table, best_score = None, 0
    for t in soup.find_all("table"):
        text = t.get_text(" ", strip=True)
        score = sum(1 for c in CATEGORIES if c in text)
        if score > best_score:
            best_table, best_score = t, score
    return best_table if best_score >= 5 else None


def parse_city_table(html: str):
    """Ritorna una lista di dict con i prezzi trovati nella pagina.

    Struttura reale delle righe (vedi HTML fornito dall'utente):
      - riga di categoria: <tr><th>...<div class="category_title">Restaurants</div>...</th>...</tr>
      - riga separatrice:  <tr class="break_category"><td colspan="3"></td></tr>
      - riga di prezzo:    <tr><td>Nome voce</td>
                                <td class="priceValue"><span class="first_currency">€18.00</span></td>
                                <td class="priceBarTd"><span class="barTextLeft">15.00</span>...
                                    <span class="barTextRight">25.00</span></td></tr>
    """
    soup = BeautifulSoup(html, "html.parser")
    table = find_price_table(soup)
    if table is None:
        return []

    rows_out = []
    current_category = None

    for tr in table.find_all("tr"):
        # 1) riga di intestazione categoria (il titolo sta in un <th>, non in <td>)
        cat_div = tr.find("div", class_="category_title")
        if cat_div is not None:
            current_category = cat_div.get_text(strip=True)
            continue

        # 2) riga separatrice tra categorie: nessun dato utile
        tr_classes = tr.get("class") or []
        if "break_category" in tr_classes:
            continue

        # 3) riga di prezzo vera e propria
        tds = tr.find_all("td")
        if len(tds) < 2 or current_category is None:
            continue

        item_name = tds[0].get_text(strip=True)
        if not item_name:
            continue

        price_span = tds[1].find("span", class_="first_currency")
        price_text = price_span.get_text(strip=True) if price_span else tds[1].get_text(strip=True)
        currency, price = parse_price(price_text)
        if price is None:
            continue

        range_min = range_max = None
        if len(tds) >= 3:
            left_span = tds[2].find("span", class_="barTextLeft")
            right_span = tds[2].find("span", class_="barTextRight")
            if left_span is not None and right_span is not None:
                try:
                    range_min = float(left_span.get_text(strip=True).replace(",", ""))
                    range_max = float(right_span.get_text(strip=True).replace(",", ""))
                except ValueError:
                    pass

        rows_out.append({
            "category": current_category,
            "item": item_name,
            "price": price,
            "currency": currency,
            "range_min": range_min,
            "range_max": range_max,
        })
    return rows_out


def main():
    session = requests.Session()

    print("Scarico l'elenco ufficiale delle città italiane presenti su Numbeo...")
    slug_map = get_city_slug_map(session)
    print(f"Trovate {len(slug_map)} città italiane su Numbeo.\n")

    long_rows = []
    not_found = []
    partial = []

    for i, province in enumerate(PROVINCE_CAPITALS, 1):
        slug = resolve_slug(province, slug_map)
        tag = f"[{i}/{len(PROVINCE_CAPITALS)}] {province}"

        if slug is None:
            print(f"{tag}: nessuna pagina trovata su Numbeo, salto.")
            not_found.append(province)
            continue

        print(f"{tag} -> {slug} ... ", end="", flush=True)
        html = fetch_city_page(session, slug)
        if html is None:
            not_found.append(province)
        else:
            items = parse_city_table(html)
            print(f"{len(items)} voci di prezzo trovate.")
            if len(items) == 0:
                not_found.append(province)
            elif len(items) < PARTIAL_THRESHOLD:
                partial.append((province, len(items)))
            for it in items:
                long_rows.append({"provincia": province, **it})

        # pausa "educata" tra una richiesta e l'altra
        time.sleep(random.uniform(*DELAY_RANGE))

    if not long_rows:
        print("\nNessun dato raccolto, controlla la connessione o la struttura del sito.")
        return

    long_df = pd.DataFrame(long_rows)

    long_df["colonna"] = long_df["category"] + " | " + long_df["item"]
    wide_df = long_df.pivot_table(
        index="provincia", columns="colonna", values="price", aggfunc="first"
    )

    # Salva sul Desktop dell'utente. Se la cartella non esiste (capita su
    # alcune configurazioni Windows/Linux) usa la cartella corrente come
    # ripiego e lo segnala chiaramente.
    desktop = Path.home() / "Desktop"
    if not desktop.is_dir():
        print(f"\nAttenzione: non trovo la cartella Desktop ({desktop}), "
              f"salvo nella cartella corrente.")
        desktop = Path(".")

    long_path = desktop / "numbeo_italia_long.csv"
    wide_path = desktop / "numbeo_italia_wide.csv"

    long_df.drop(columns="colonna").to_csv(long_path, index=False, encoding="utf-8-sig")
    wide_df.to_csv(wide_path, encoding="utf-8-sig")

    print("\nCompletato.")
    print("File creati:")
    print(f"  - {long_path}  (una riga per ogni prezzo)")
    print(f"  - {wide_path}  (una riga per provincia, una colonna per prezzo)")

    if not_found:
        print("\nCittà non trovate/senza alcun dato su Numbeo (verifica a mano il nome esatto):")
        for p in not_found:
            print(" -", p)

    if partial:
        print(f"\nCittà con meno di {PARTIAL_THRESHOLD} voci di prezzo trovate "
              f"(dati incompleti su Numbeo, non un errore dello script):")
        for p, n in partial:
            print(f" - {p}: {n} voci")


if __name__ == "__main__":
    main()

Scarico l'elenco ufficiale delle città italiane presenti su Numbeo...


HTTPError: 429 Client Error:  for url: https://www.numbeo.com/cost-of-living/country_result.jsp?country=Italy

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Numbeo Cost of Living Scraper - Capoluoghi di provincia italiani
===================================================================

COSA FA
-------
Scarica, una pagina alla volta, i dati di "Cost of Living" da numbeo.com
per i capoluoghi delle 110 province italiane (configurazione storica a
110 province) e produce due file CSV:

  - numbeo_italia_long.csv  -> formato "lungo": una riga per ogni singolo
                                prezzo (provincia, categoria, voce, prezzo, range)
  - numbeo_italia_wide.csv  -> formato "tabellare": una riga per provincia,
                                una colonna per ogni voce di prezzo
                                (es. "Restaurants | Cappuccino (Regular Size)")

IMPORTANTE - LEGGERE PRIMA DI USARE
------------------------------------
I Termini di Uso di Numbeo (numbeo.com/common/terms_of_use.jsp) vietano
lo scraping/crawling automatico senza permesso scritto, ma consentono
l'uso libero dei DATI per scopi personali, a patto di citare la fonte
(link a numbeo.com). Questo script è pensato quindi per:
  - uso PERSONALE (non commerciale, non ridistribuzione dei dati grezzi);
  - un ritmo di richieste deliberatamente lento (una richiesta alla volta,
    pausa di alcuni secondi tra una città e l'altra, niente parallelismo);
  - citare sempre numbeo.com come fonte se pubblichi qualcosa basato su
    questi dati.
Per usi commerciali, massivi o ripetuti nel tempo, Numbeo vende un Data
License / API ufficiale: numbeo.com/common/api.jsp

REQUISITI
---------
    pip install requests beautifulsoup4 pandas

USO
---
    python numbeo_scraper_province_italiane.py

    Per riprovare solo le province mancate al primo giro (elenco in
    RETRY_PROVINCES, da aggiornare a mano ad ogni tentativo), i risultati
    vengono uniti ai CSV già presenti sul Desktop invece di sovrascriverli:

    python numbeo_scraper_province_italiane.py --retry
"""

import re
import sys
import time
import random
import unicodedata
import difflib
from pathlib import Path
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.numbeo.com"
COUNTRY_URL = f"{BASE}/cost-of-living/country_result.jsp?country=Italy"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0 Safari/537.36 personal-research-script"
}

# Pausa (in secondi) tra una richiesta e l'altra: mantienila alta per
# rispettare il server. Non ridurla per "velocizzare" lo scraping.
DELAY_RANGE = (4, 8)

# Sotto questa soglia di voci trovate, la provincia viene segnalata a fine
# esecuzione come "dati incompleti" — non è un errore dello script, significa
# solo che su Numbeo quella città ha poche informazioni inserite dagli utenti.
PARTIAL_THRESHOLD = 30

# ---------------------------------------------------------------------------
# 110 capoluoghi di provincia italiani (configurazione storica a 110 province,
# comprese le 4 province sarde poi riassorbite). Modifica liberamente questa
# lista se vuoi usare la configurazione attuale (107 enti) o un sottoinsieme.
# ---------------------------------------------------------------------------
PROVINCE_CAPITALS = [
    # Piemonte
    "Torino", "Vercelli", "Novara", "Cuneo", "Asti", "Alessandria", "Biella", "Verbania",
    # Valle d'Aosta
    "Aosta",
    # Lombardia
    "Varese", "Como", "Sondrio", "Milano", "Bergamo", "Brescia", "Pavia", "Cremona",
    "Mantova", "Lecco", "Lodi", "Monza",
    # Trentino-Alto Adige
    "Bolzano", "Trento",
    # Veneto
    "Verona", "Vicenza", "Belluno", "Treviso", "Venezia", "Padova", "Rovigo",
    # Friuli-Venezia Giulia
    "Udine", "Gorizia", "Trieste", "Pordenone",
    # Liguria
    "Imperia", "Savona", "Genova", "La Spezia",
    # Emilia-Romagna
    "Piacenza", "Parma", "Reggio Emilia", "Modena", "Bologna", "Ferrara", "Ravenna",
    "Forli", "Rimini",
    # Toscana
    "Massa", "Lucca", "Pistoia", "Firenze", "Livorno", "Pisa", "Arezzo", "Siena",
    "Grosseto", "Prato",
    # Umbria
    "Perugia", "Terni",
    # Marche
    "Pesaro", "Ancona", "Macerata", "Ascoli Piceno", "Fermo",
    # Lazio
    "Viterbo", "Rieti", "Roma", "Latina", "Frosinone",
    # Abruzzo
    "L'Aquila", "Teramo", "Pescara", "Chieti",
    # Molise
    "Campobasso", "Isernia",
    # Campania
    "Caserta", "Benevento", "Napoli", "Avellino", "Salerno",
    # Puglia
    "Foggia", "Bari", "Taranto", "Brindisi", "Lecce", "Barletta",
    # Basilicata
    "Potenza", "Matera",
    # Calabria
    "Cosenza", "Catanzaro", "Reggio Calabria", "Crotone", "Vibo Valentia",
    # Sicilia
    "Trapani", "Palermo", "Messina", "Agrigento", "Caltanissetta", "Enna", "Catania",
    "Ragusa", "Siracusa",
    # Sardegna (incluse le 4 ex-province poi riassorbite, per arrivare a 110)
    "Sassari", "Nuoro", "Cagliari", "Oristano", "Olbia", "Lanusei", "Sanluri", "Carbonia",
]

# ---------------------------------------------------------------------------
# Province da riprovare (mancate nella prima scrapata). Il nome a sinistra
# nel commento è quello "ufficiale" della provincia, quello nella lista è il
# nome del capoluogo che usa PROVINCE_CAPITALS/lo script per cercarlo su
# Numbeo. Aggiorna liberamente questa lista ad ogni nuovo tentativo.
# ---------------------------------------------------------------------------
RETRY_PROVINCES = [
    "Verbania",        # Verbano-Cusio-Ossola
    "Imperia",
    "La Spezia",
    "Sondrio",
    "Rovigo",
    "Udine",
    "Massa",           # Massa-Carrara
    "Pesaro",          # Pesaro e Urbino
    "Ascoli Piceno",
    "Viterbo",
    "Frosinone",
    "Campobasso",
    "Isernia",
    "Benevento",
    "Avellino",
    "Potenza",
    "Matera",
    "Catanzaro",
    "Crotone",
    "Vibo Valentia",   # attenzione: probabilmente senza pagina su Numbeo
    "Trapani",
    "Messina",
    "Agrigento",
    "Caltanissetta",   # attenzione: probabilmente senza pagina su Numbeo
    "Enna",            # attenzione: probabilmente senza pagina su Numbeo
    "Ragusa",
    "Siracusa",
    "Nuoro",
    "Oristano",
]

# Alias manuali per i casi in cui Numbeo usa un nome diverso da quello
# "ufficiale" italiano (nomi inglesi per le città maggiori, ecc.)
MANUAL_ALIASES = {
    "torino": "Turin",
    "milano": "Milan",
    "firenze": "Florence",
    "genova": "Genoa",
    "napoli": "Naples",
    "roma": "Rome",
    "venezia": "Venice",
    "reggio emilia": "Reggio Nell'emilia",
    "reggio calabria": "Reggio Di Calabria",
    "bolzano": "Bolzano-Bozen",
}

# Le 10 categorie di prezzo che ci interessano (nomi esatti come compaiono
# sulla pagina di Numbeo)
CATEGORIES = [
    "Restaurants", "Markets", "Transportation", "Utilities (Monthly)",
    "Sports And Leisure", "Childcare", "Clothing And Shoes",
    "Rent Per Month", "Buy Apartment Price", "Salaries And Financing",
]

PRICE_RE = re.compile(r"([€$£])\s*([\d.,]+)")
NUMBER_RE = re.compile(r"[\d][\d.,]*")


def normalize(s: str) -> str:
    """Normalizza un nome città per il confronto: minuscolo, senza accenti,
    solo lettere/numeri."""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]", "", s.lower())


def get_city_slug_map(session: requests.Session) -> dict:
    """Scarica la pagina 'Italy' di Numbeo e legge il menu a tendina delle
    città per ottenere la mappa {nome_normalizzato: slug_url_reale}.
    Questo evita di dover indovinare a mano l'URL di ogni città."""
    r = session.get(COUNTRY_URL, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    select = None
    for sel in soup.find_all("select"):
        opts = sel.find_all("option")
        if opts and "select city" in opts[0].get_text(strip=True).lower():
            select = sel
            break
    if select is None:
        raise RuntimeError(
            "Non trovo il menu a tendina delle città nella pagina Italia. "
            "La struttura del sito potrebbe essere cambiata."
        )

    mapping = {}
    for opt in select.find_all("option"):
        label = opt.get_text(strip=True)
        if not label or "select city" in label.lower():
            continue
        value = (opt.get("value") or label).strip()
        mapping[normalize(label)] = value
    return mapping


def resolve_slug(city_name: str, slug_map: dict):
    """Trova lo slug URL corretto per una data città."""
    alias = MANUAL_ALIASES.get(city_name.lower())
    key = normalize(alias) if alias else normalize(city_name)
    if key in slug_map:
        return slug_map[key]
    close = difflib.get_close_matches(key, slug_map.keys(), n=1, cutoff=0.8)
    if close:
        return slug_map[close[0]]
    return None


def fetch_city_page(session: requests.Session, slug: str, max_retries: int = 3):
    url = f"{BASE}/cost-of-living/in/{quote(slug, safe='')}"
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=30)
        except requests.RequestException as e:
            print(f"errore di rete ({e}), tentativo {attempt}/{max_retries} ... ", end="", flush=True)
            time.sleep(random.uniform(6, 12))
            continue

        if r.status_code == 200:
            return r.text
        if r.status_code in (429, 503):
            wait = 15 * attempt
            print(f"HTTP {r.status_code} (probabile blocco temporaneo), "
                  f"aspetto {wait}s e riprovo ({attempt}/{max_retries}) ... ", end="", flush=True)
            time.sleep(wait)
            continue

        print(f"HTTP {r.status_code}")
        return None

    print("falliti tutti i tentativi")
    return None


def parse_price(text: str):
    """Estrae (valuta, valore) da un testo tipo '€18.00'. Alcune voci
    (es. il tasso di interesse ipotecario) non hanno simbolo di valuta,
    in quel caso ritorna (None, valore)."""
    m = PRICE_RE.search(text)
    if m:
        try:
            return m.group(1), float(m.group(2).replace(",", ""))
        except ValueError:
            return m.group(1), None
    m2 = NUMBER_RE.search(text)
    if m2:
        try:
            return None, float(m2.group(0).replace(",", ""))
        except ValueError:
            return None, None
    return None, None


def find_price_table(soup: BeautifulSoup):
    """Individua la tabella dei prezzi. Numbeo la marca con la classe
    'data_wide_table'; in fallback cerca la tabella che contiene il
    maggior numero di intestazioni di categoria note (utile se cambiano
    le classi CSS in futuro)."""
    table = soup.find("table", class_="data_wide_table")
    if table is not None:
        return table

    best_table, best_score = None, 0
    for t in soup.find_all("table"):
        text = t.get_text(" ", strip=True)
        score = sum(1 for c in CATEGORIES if c in text)
        if score > best_score:
            best_table, best_score = t, score
    return best_table if best_score >= 5 else None


def parse_city_table(html: str):
    """Ritorna una lista di dict con i prezzi trovati nella pagina.

    Struttura reale delle righe (vedi HTML fornito dall'utente):
      - riga di categoria: <tr><th>...<div class="category_title">Restaurants</div>...</th>...</tr>
      - riga separatrice:  <tr class="break_category"><td colspan="3"></td></tr>
      - riga di prezzo:    <tr><td>Nome voce</td>
                                <td class="priceValue"><span class="first_currency">€18.00</span></td>
                                <td class="priceBarTd"><span class="barTextLeft">15.00</span>...
                                    <span class="barTextRight">25.00</span></td></tr>
    """
    soup = BeautifulSoup(html, "html.parser")
    table = find_price_table(soup)
    if table is None:
        return []

    rows_out = []
    current_category = None

    for tr in table.find_all("tr"):
        # 1) riga di intestazione categoria (il titolo sta in un <th>, non in <td>)
        cat_div = tr.find("div", class_="category_title")
        if cat_div is not None:
            current_category = cat_div.get_text(strip=True)
            continue

        # 2) riga separatrice tra categorie: nessun dato utile
        tr_classes = tr.get("class") or []
        if "break_category" in tr_classes:
            continue

        # 3) riga di prezzo vera e propria
        tds = tr.find_all("td")
        if len(tds) < 2 or current_category is None:
            continue

        item_name = tds[0].get_text(strip=True)
        if not item_name:
            continue

        price_span = tds[1].find("span", class_="first_currency")
        price_text = price_span.get_text(strip=True) if price_span else tds[1].get_text(strip=True)
        currency, price = parse_price(price_text)
        if price is None:
            continue

        range_min = range_max = None
        if len(tds) >= 3:
            left_span = tds[2].find("span", class_="barTextLeft")
            right_span = tds[2].find("span", class_="barTextRight")
            if left_span is not None and right_span is not None:
                try:
                    range_min = float(left_span.get_text(strip=True).replace(",", ""))
                    range_max = float(right_span.get_text(strip=True).replace(",", ""))
                except ValueError:
                    pass

        rows_out.append({
            "category": current_category,
            "item": item_name,
            "price": price,
            "currency": currency,
            "range_min": range_min,
            "range_max": range_max,
        })
    return rows_out


def main():
    retry_mode = "--retry" in sys.argv
    provinces = RETRY_PROVINCES if retry_mode else PROVINCE_CAPITALS
    # in modalità retry andiamo più piano: probabilmente le mancate la prima
    # volta sono saltate per un blocco temporaneo dovuto alla frequenza
    delay_range = (10, 18) if retry_mode else DELAY_RANGE

    if retry_mode:
        print(f"Modalità RETRY: riprovo solo le {len(provinces)} province mancanti.\n")

    session = requests.Session()

    print("Scarico l'elenco ufficiale delle città italiane presenti su Numbeo...")
    slug_map = get_city_slug_map(session)
    print(f"Trovate {len(slug_map)} città italiane su Numbeo.\n")

    long_rows = []
    not_found = []
    partial = []

    for i, province in enumerate(provinces, 1):
        slug = resolve_slug(province, slug_map)
        tag = f"[{i}/{len(provinces)}] {province}"

        if slug is None:
            print(f"{tag}: nessuna pagina trovata su Numbeo, salto.")
            not_found.append(province)
            continue

        print(f"{tag} -> {slug} ... ", end="", flush=True)
        html = fetch_city_page(session, slug)
        if html is None:
            not_found.append(province)
        else:
            items = parse_city_table(html)
            print(f"{len(items)} voci di prezzo trovate.")
            if len(items) == 0:
                not_found.append(province)
            elif len(items) < PARTIAL_THRESHOLD:
                partial.append((province, len(items)))
            for it in items:
                long_rows.append({"provincia": province, **it})

        # pausa "educata" tra una richiesta e l'altra
        time.sleep(random.uniform(*delay_range))

    desktop = Path.home() / "Desktop"
    if not desktop.is_dir():
        print(f"\nAttenzione: non trovo la cartella Desktop ({desktop}), "
              f"salvo nella cartella corrente.")
        desktop = Path(".")

    long_path = desktop / "numbeo_italia_long.csv"
    wide_path = desktop / "numbeo_italia_wide.csv"

    long_df = pd.DataFrame(long_rows)

    if retry_mode and long_path.exists():
        print(f"\nUnisco i nuovi dati con quelli già presenti in {long_path} ...")
        existing_df = pd.read_csv(long_path, encoding="utf-8-sig")
        # tolgo dal file esistente le province appena riprovate, cosi' se
        # avevano dati parziali vengono sostituiti da quelli nuovi (non duplicati)
        existing_df = existing_df[~existing_df["provincia"].isin(provinces)]
        long_df = pd.concat([existing_df, long_df], ignore_index=True)
    elif not len(long_df):
        print("\nNessun dato raccolto, controlla la connessione o la struttura del sito.")
        return

    long_df["colonna"] = long_df["category"] + " | " + long_df["item"]
    wide_df = long_df.pivot_table(
        index="provincia", columns="colonna", values="price", aggfunc="first"
    )

    long_df.drop(columns="colonna").to_csv(long_path, index=False, encoding="utf-8-sig")
    wide_df.to_csv(wide_path, encoding="utf-8-sig")

    print("\nCompletato.")
    print("File aggiornati:" if retry_mode else "File creati:")
    print(f"  - {long_path}  (una riga per ogni prezzo)")
    print(f"  - {wide_path}  (una riga per provincia, una colonna per prezzo)")

    if not_found:
        print("\nCittà non trovate/senza alcun dato su Numbeo (verifica a mano il nome esatto):")
        for p in not_found:
            print(" -", p)

    if partial:
        print(f"\nCittà con meno di {PARTIAL_THRESHOLD} voci di prezzo trovate "
              f"(dati incompleti su Numbeo, non un errore dello script):")
        for p, n in partial:
            print(f" - {p}: {n} voci")


if __name__ == "__main__":
    main()